# Freemarket Medallion Pipeline — orchestrator

This notebook **orchestrates and narrates**; all transformation logic lives in the tested
`src/` package (see `plan/SPEC.md` § Engineering Standards). Run top-to-bottom
(Restart & Run All) from an empty warehouse.

**Ticket: Pipeline foundation** — open the warehouse and create the six Medallion schemas
(`raw`, `live`, `core`, `shape`, `data_mart`, `curated`).

In [1]:
# Make the repo root importable when running from notebook/
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, warehouse
from src.pipeline import build_foundation
config.WAREHOUSE_PATH

PosixPath('/Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/submission/warehouse.duckdb')

## Build the foundation
Connect (creates the file if absent) and create the schemas. Idempotent.

In [2]:
con = warehouse.connect()
created = build_foundation(con)
created

17:50:11 | INFO  | [foundation] schemas ready: raw, live, core, shape, data_mart, curated


('raw', 'live', 'core', 'shape', 'data_mart', 'curated')

## Verify — all six schemas present

In [3]:
present = warehouse.existing_schemas(con)
assert set(config.SCHEMAS) <= present, f'missing: {set(config.SCHEMAS) - present}'
con.sql("""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name IN ('raw','live','core','shape','data_mart','curated')
    ORDER BY schema_name
""").df()

,schema_name
0,core
1,curated
2,data_mart
3,live
4,raw
5,shape


In [4]:
con.close()
print('Foundation ready — six schemas created in', config.WAREHOUSE_PATH)

Foundation ready — six schemas created in /Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/submission/warehouse.duckdb
